# Colab Setup (run first)
This notebook supports both local VS Code and Google Colab.

- If running on Colab, run the next setup cell first.
- Set `PROJECT_ROOT` if your repo is in a custom location.
- Optional: mount Google Drive if your data is stored there.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Install notebook-only dependencies in Colab runtime.
    %pip -q install rank-bm25 sentence-transformers

    from google.colab import drive
    drive.mount("/content/drive")
        

# Optional override if your repo root is non-standard.
# Example: /content/drive/MyDrive/Legal-question-answer-v1
PROJECT_ROOT = os.environ.get("PROJECT_ROOT", "")
if PROJECT_ROOT:
    print(f"Using PROJECT_ROOT from environment: {PROJECT_ROOT}")

In [2]:
import sys
from pathlib import Path

# Resolve project root in both local and Colab environments.
candidate_roots = []

# 1) Explicit env override from setup cell
project_root_env = Path(PROJECT_ROOT).expanduser() if "PROJECT_ROOT" in globals() and PROJECT_ROOT else None
if project_root_env:
    candidate_roots.append(project_root_env)

# 2) Current working directory search upward
cwd = Path.cwd().resolve()
candidate_roots.extend([cwd, *cwd.parents])

# 3) Common Colab locations
candidate_roots.extend([
    Path("/content/drive/MyDrive/Colab Notebooks/NLP/Legal-question-answer-v1"),
])

project_root = None
for path in candidate_roots:
    if (path / "utils").exists() and (path / "data").exists():
        project_root = path
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not find project root containing 'utils/' and 'data/'. "
        "Set PROJECT_ROOT env var, or update the Colab setup cell path."
    )

utils_path = str(project_root / "utils")
if utils_path not in sys.path:
    sys.path.append(utils_path)

print(f"Resolved project_root: {project_root}")

from get_jsonl_data import get_jsonl_data

Resolved project_root: E:\AI\NLP\Legal-question-answer-v1


In [6]:
corpus_path = project_root / "data" / "corpus.jsonl"
chunk_store_path = project_root / "data" / "chunk_store.jsonl"

corpus = get_jsonl_data(str(corpus_path))
chunk_store = get_jsonl_data(str(chunk_store_path))

print(corpus[0])
print(chunk_store[0])

{'doc_id': 'doc_0', 'article_id': 'article_0', 'article_content': 'Điều 5. Nguyên tắc hội nhập và hợp tác quốc tế về địa chất, khoáng sản\n\n1. Hội nhập và hợp tác quốc tế trong nghiên cứu, điều tra cơ bản địa chất, điều tra địa chất về khoáng sản, hoạt động khoáng sản, quản lý hoạt động khoáng sản phải đặt trong tổng thể chiến lược phát triển kinh tế - xã hội của đất nước trong từng thời kỳ; chiến lược địa chất, khoáng sản và công nghiệp khai khoáng; tuân thủ Hiến pháp, pháp luật Việt Nam, Hiến chương Liên hợp quốc, điều ước quốc tế mà nước Cộng hòa xã hội chủ nghĩa Việt Nam là thành viên, bảo đảm phù hợp với đường lối và chính sách đối ngoại của Việt Nam; bảo đảm nguyên tắc hợp tác bình đẳng, cùng có lợi trên cơ sở tôn trọng độc lập, chủ quyền và toàn vẹn lãnh thổ, không can thiệp vào công việc nội bộ của nhau.\n\n2. Tranh chấp quốc tế về địa chất, khoáng sản được giải quyết thông qua các biện pháp hòa bình, theo thông lệ quốc tế, pháp luật quốc tế và pháp luật của các bên liên quan.

In [ ]:
max_num_of_chunk = 0
temp = "" 
count = 0
max_article_id = ""
for chunk in chunk_store:
    article_id = chunk.get("article_id")
    if article_id == temp:
        count += 1
    else:
        if count > max_num_of_chunk:
            max_num_of_chunk = count
            max_article_id = temp

        temp = article_id
        count = 1

print(f"Max number of chunks of a article: {max_num_of_chunk}")
print(f"Article has the largest number of chunks: {max_article_id}")

In [ ]:
def bm25_retrieve_top_k(question, chunks, top_k=20):
    if not chunks:
        return []

    tokenized_chunks = [chunk.split() for chunk in chunks]
    bm25 = BM25Okapi(tokenized_chunks)

    tokenized_question = question.split()
    scores = bm25.get_scores(tokenized_question)

    top_k = min(top_k, len(chunks))
    if top_k <= 0:
        return []

    # Faster than full argsort when top_k << len(chunks).
    if top_k < len(chunks):
        top_idx = np.argpartition(scores, -top_k)[-top_k:]
        top_k_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
    else:
        top_k_idx = np.argsort(scores)[::-1]

    return [chunks[i] for i in top_k_idx]


def truncate_pair_for_cross_encoder(question, chunk, tokenizer, max_length=256, q_max_tokens=64):
    # Keep pair length bounded so long legal chunks do not overflow tokenizer limits.
    q_ids = tokenizer.encode(
        question,
        add_special_tokens=False,
        truncation=True,
        max_length=q_max_tokens,
    )

    special_tokens = tokenizer.num_special_tokens_to_add(pair=True)
    chunk_budget = max(8, max_length - len(q_ids) - special_tokens)

    c_ids = tokenizer.encode(
        chunk,
        add_special_tokens=False,
        truncation=True,
        max_length=chunk_budget,
    )

    short_q = tokenizer.decode(q_ids, skip_special_tokens=True)
    short_c = tokenizer.decode(c_ids, skip_special_tokens=True)
    return short_q, short_c


def rerank_with_phobert(
    question,
    candidate_chunks,
    reranker,
    top_n=5,
    max_length=256,
    batch_size=64,
    show_progress_bar=False,
    ):
    if not candidate_chunks:
        return []

    # Build (question, chunk) pairs after truncation, then score with cross-encoder.
    pairs = [
        list(truncate_pair_for_cross_encoder(question, chunk, reranker.tokenizer, max_length=max_length))
        for chunk in candidate_chunks
    ]

    scores = reranker.predict(
        pairs,
        batch_size=batch_size,
        show_progress_bar=show_progress_bar,
    )
    ranked = sorted(
        zip(candidate_chunks, scores),
        key=lambda x: x[1],
        reverse=True,
    )

    return [
        {"chunk": chunk, "cross_score": float(score)}
        for chunk, score in ranked[:top_n]
    ]

In [ ]:
from collections import defaultdict
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from transformers import logging as hf_logging
import numpy as np
import time

# Reduce noisy tokenizer warnings in Colab logs.
hf_logging.set_verbosity_error()

# Vietnamese PhoBERT-based cross-encoder reranker
phobert_model_name = "itdainb/PhoRanker"
reranker = CrossEncoder(phobert_model_name, max_length=256)

hard_neg_k = 5
log_every_docs = 20
log_every_questions = 200

# Adaptive retrieval/rerank config
bm25_base_k = 50
bm25_max_k = 2000
bm25_pos_multiplier = 2.0
bm25_extra_margin = 20
bm25_max_pos_multiplier = 3.0
bm25_max_extra_margin = 50
bm25_hard_cap = 6000

rerank_start_k = 30
rerank_max_k = 300
rerank_growth = 2.0
rerank_start_pos_multiplier = 0.2
rerank_start_extra_margin = 10
# For positive_count = 1500, adaptive_rerank_max_k ~= 1500
rerank_max_pos_multiplier = 1.0
rerank_max_extra_margin = 0
rerank_hard_cap = 1800
rerank_batch_size = 64
rerank_fallback_step = 200

# Pre-index once:
# 1) chunks_by_doc for candidate retrieval in each document
# 2) chunks_by_article for true positive chunks by (doc_id, article_id)
chunks_by_doc = defaultdict(list)
chunks_by_article = defaultdict(list)
for chunk in chunk_store:
    doc = chunk.get("doc_id")
    art = chunk.get("article_id")
    text = chunk.get("text", "").strip()
    if not doc or not text:
        continue
    chunks_by_doc[doc].append(text)
    if art is not None:
        chunks_by_article[(doc, art)].append(text)

all_hard_negative_samples = []
total_docs = len(corpus)
processed_questions = 0
insufficient_neg_count = 0
start_time = time.time()

print(f"Indexed chunks for {len(chunks_by_doc)} docs")
print(f"Indexed article-level positives for {len(chunks_by_article)} doc/article pairs")
print(f"Start processing {total_docs} corpus rows...")

for doc_idx, row in enumerate(corpus, start=1):
    doc_id = row["doc_id"]
    article_id = row.get("article_id")

    qa_pairs = row.get("generated_qa_pairs", [])
    chunk_texts = chunks_by_doc.get(doc_id, [])

    # Positive chunks are all chunks from the same (doc_id, article_id).
    positive_chunks = chunks_by_article.get((doc_id, article_id), [])
    positive_set = set(positive_chunks)

    if not chunk_texts or not qa_pairs:
        if doc_idx % log_every_docs == 0 or doc_idx == total_docs:
            elapsed = time.time() - start_time
            print(
                f"Docs: {doc_idx}/{total_docs} | Samples: {len(all_hard_negative_samples)} "
                f"| Questions: {processed_questions} | InsufficientNeg: {insufficient_neg_count} "
                f"| Elapsed: {elapsed:.1f}s"
            )
        continue

    # Build BM25 once per document, reuse for all questions in that document.
    tokenized_chunks = [chunk.split() for chunk in chunk_texts]
    bm25 = BM25Okapi(tokenized_chunks)

    for qa_pair in qa_pairs:
        question = qa_pair.get("question", "").strip()
        if not question:
            continue

        tokenized_question = question.split()
        scores = bm25.get_scores(tokenized_question)

        positive_count = len(positive_set)

        # Compute BM25 candidate size adaptively from positive size, with safety cap.
        target_bm25_k = int(
            max(
                bm25_base_k,
                positive_count * bm25_pos_multiplier + bm25_extra_margin,
            )
        )
        adaptive_bm25_max_k = int(
            max(
                bm25_max_k,
                positive_count * bm25_max_pos_multiplier + bm25_max_extra_margin,
            )
        )
        adaptive_bm25_max_k = min(adaptive_bm25_max_k, bm25_hard_cap)
        bm25_k = min(len(chunk_texts), adaptive_bm25_max_k, target_bm25_k)
        if bm25_k <= 0:
            bm25_k = min(len(chunk_texts), bm25_base_k)

        # Partial top-k is faster than fully sorting all BM25 scores.
        if bm25_k < len(chunk_texts):
            top_idx = np.argpartition(scores, -bm25_k)[-bm25_k:]
            sorted_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
        else:
            sorted_idx = np.argsort(scores)[::-1]

        candidate_chunks = [chunk_texts[i] for i in sorted_idx[:bm25_k]]

        hard_negatives = []
        adaptive_rerank_max_k = int(
            max(
                rerank_max_k,
                positive_count * rerank_max_pos_multiplier + rerank_max_extra_margin,
            )
        )
        adaptive_rerank_max_k = min(adaptive_rerank_max_k, rerank_hard_cap)
        adaptive_phase_max_k = min(adaptive_rerank_max_k, len(candidate_chunks))

        adaptive_rerank_start_k = int(
            max(
                rerank_start_k,
                positive_count * rerank_start_pos_multiplier + rerank_start_extra_margin,
            )
        )
        current_rerank_k = min(adaptive_rerank_start_k, adaptive_phase_max_k)

        # Two-phase rerank: adaptive phase (fast) then fallback expansion (guarantee).
        # Guarantee is over current BM25 candidate pool, without pre-filtering positives.
        guarantee_max_rerank_k = len(candidate_chunks)

        # Incremental rerank: each candidate is scored once, then reused in later rounds.
        scored_until = 0
        scored_items = []

        while current_rerank_k > 0 and current_rerank_k <= guarantee_max_rerank_k:
            if current_rerank_k > scored_until:
                new_candidates = candidate_chunks[scored_until:current_rerank_k]
                new_scored = rerank_with_phobert(
                    question=question,
                    candidate_chunks=new_candidates,
                    reranker=reranker,
                    top_n=len(new_candidates),
                    max_length=256,
                    batch_size=rerank_batch_size,
                    show_progress_bar=False,
                )
                scored_items.extend(new_scored)
                scored_until = current_rerank_k

            reranked_chunks = sorted(
                scored_items,
                key=lambda x: x["cross_score"],
                reverse=True,
            )

            # Keep only non-positive chunks as hard negatives.
            hard_negatives = [
                item["chunk"]
                for item in reranked_chunks
                if item["chunk"] not in positive_set
            ][:hard_neg_k]

            if len(hard_negatives) >= hard_neg_k:
                break

            if current_rerank_k == guarantee_max_rerank_k:
                break

            if current_rerank_k < adaptive_phase_max_k:
                next_k = int(current_rerank_k * rerank_growth)
                if next_k <= current_rerank_k:
                    next_k = current_rerank_k + 1
                current_rerank_k = min(next_k, adaptive_phase_max_k)
            else:
                # Fallback expansion for guarantee: continue beyond adaptive cap.
                current_rerank_k = min(
                    current_rerank_k + rerank_fallback_step,
                    guarantee_max_rerank_k,
                )

        if len(hard_negatives) < hard_neg_k:
            insufficient_neg_count += 1

        all_hard_negative_samples.append(
            {
                "doc_id": doc_id,
                "article_id": article_id,
                "question": question,
                "positive_chunks": positive_chunks,
                "negative_chunks": hard_negatives,
            }
        )

        processed_questions += 1
        if processed_questions % log_every_questions == 0:
            elapsed = time.time() - start_time
            print(
                f"Questions: {processed_questions} | Docs: {doc_idx}/{total_docs} "
                f"| Samples: {len(all_hard_negative_samples)} | InsufficientNeg: {insufficient_neg_count} "
                f"| Elapsed: {elapsed:.1f}s"
            )

    if doc_idx % log_every_docs == 0 or doc_idx == total_docs:
        elapsed = time.time() - start_time
        print(
            f"Docs: {doc_idx}/{total_docs} | Samples: {len(all_hard_negative_samples)} "
            f"| Questions: {processed_questions} | InsufficientNeg: {insufficient_neg_count} "
            f"| Elapsed: {elapsed:.1f}s"
        )

elapsed = time.time() - start_time
print(f"Collected {len(all_hard_negative_samples)} question-level samples in {elapsed:.1f}s")
print(f"Questions with <{hard_neg_k} negatives: {insufficient_neg_count}")
print(all_hard_negative_samples[0] if all_hard_negative_samples else "No samples")

In [ ]:
import json

output_path = project_root / "data" / "retriever_hard_negatives_phobert.jsonl"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    for item in all_hard_negative_samples:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved to: {output_path}")